In [ ]:
# Setup and Formatting
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from skimage.metrics import structural_similarity as ssim
import time

plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'


In [ ]:
# 1. Dataset Loading
print("Loading dataset...")
(x_train, _), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:10000].astype('float32') / 255.
x_test = x_test[:2000].astype('float32') / 255.
y_test = y_test[:2000]

x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)
x_train_conv = np.expand_dims(x_train, -1)
x_test_conv = np.expand_dims(x_test, -1)

def compute_metrics(original, reconstructed):
    mse = np.mean((original - reconstructed)**2)
    mae = np.mean(np.abs(original - reconstructed))
    ssim_vals = [ssim(original[i], reconstructed[i], data_range=1.0) for i in range(len(original))]
    mean_ssim = np.mean(ssim_vals)
    return mse, mae, mean_ssim


In [ ]:
# --- Model 1: Fully Connected Autoencoder ---
input_flat = layers.Input(shape=(784,))
encoded_fc = layers.Dense(128, activation='relu')(input_flat)
encoded_fc = layers.Dense(32, activation='relu')(encoded_fc)
latent_fc = layers.Dense(16, activation='relu')(encoded_fc)
decoded_fc = layers.Dense(32, activation='relu')(latent_fc)
decoded_fc = layers.Dense(128, activation='relu')(decoded_fc)
output_fc = layers.Dense(784, activation='sigmoid')(decoded_fc)

fc_ae = models.Model(input_flat, output_fc)
fc_ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy')

print("Training FC Autoencoder...")
start_time = time.time()
history_fc = fc_ae.fit(x_train_flat, x_train_flat, epochs=20, batch_size=128, shuffle=True, validation_data=(x_test_flat, x_test_flat), verbose=1)
print(f"FC AE trained in {time.time() - start_time:.2f}s")


In [ ]:
# Plot 1 & 2 for FC Autoencoder
decoded_imgs_fc = fc_ae.predict(x_test_flat).reshape(-1, 28, 28)
fc_mse, fc_mae, fc_ssim = compute_metrics(x_test, decoded_imgs_fc)
print(f"FC AE Metrics: MSE={fc_mse:.4f}, MAE={fc_mae:.4f}, SSIM={fc_ssim:.4f}")

fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(10):
    axes[0, i].imshow(x_test[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(decoded_imgs_fc[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
plt.savefig('plot1_fc_recon.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(history_fc.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history_fc.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Reconstruction Loss', fontweight='bold')
plt.legend(prop={'weight': 'bold'})
plt.savefig('plot2_fc_loss.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# --- Model 2: Convolutional Autoencoder ---
input_conv = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', padding='same')(input_conv)
x = layers.MaxPooling2D(2, padding='same')(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D(2, padding='same')(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
encoded_conv = x

x = layers.UpSampling2D(2)(encoded_conv)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.UpSampling2D(2)(x)
decoded_conv = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

conv_ae = models.Model(input_conv, decoded_conv)
conv_ae.compile(optimizer='adam', loss='mse')

print("Training Conv Autoencoder...")
start_time = time.time()
history_conv = conv_ae.fit(x_train_conv, x_train_conv, epochs=20, batch_size=128, shuffle=True, validation_data=(x_test_conv, x_test_conv), verbose=1)
print(f"Conv AE trained in {time.time() - start_time:.2f}s")


In [ ]:
# Plot 3 for Conv Autoencoder
decoded_imgs_conv = conv_ae.predict(x_test_conv).reshape(-1, 28, 28)
conv_mse, conv_mae, conv_ssim = compute_metrics(x_test, decoded_imgs_conv)
print(f"Conv AE Metrics: MSE={conv_mse:.4f}, MAE={conv_mae:.4f}, SSIM={conv_ssim:.4f}")

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for i in range(10):
    axes[0, i].imshow(x_test[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(decoded_imgs_fc[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    axes[2, i].imshow(decoded_imgs_conv[i], cmap='gray', vmin=0, vmax=1)
    axes[2, i].axis('off')
plt.savefig('plot3_recon_compare.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# --- Model 3: Denoising Autoencoder ---
noise_factor = 0.2
x_train_noisy = np.clip(x_train_conv + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train_conv.shape), 0., 1.)
x_test_noisy = np.clip(x_test_conv + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test_conv.shape), 0., 1.)

denoise_ae = models.clone_model(conv_ae)
denoise_ae.compile(optimizer='adam', loss='mse')

print("Training Denoising Autoencoder...")
start_time = time.time()
denoise_ae.fit(x_train_noisy, x_train_conv, epochs=20, batch_size=128, shuffle=True, validation_data=(x_test_noisy, x_test_conv), verbose=1)
print(f"Denoising AE trained in {time.time() - start_time:.2f}s")


In [ ]:
# Plot 4 & 5 for Denoising Autoencoder
decoded_imgs_denoise = denoise_ae.predict(x_test_noisy).reshape(-1, 28, 28)
denoise_mse, denoise_mae, denoise_ssim = compute_metrics(x_test, decoded_imgs_denoise)
print(f"Denoising AE Metrics: MSE={denoise_mse:.4f}, MAE={denoise_mae:.4f}, SSIM={denoise_ssim:.4f}")

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for i in range(10):
    axes[0, i].imshow(x_test[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(x_test_noisy[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    axes[2, i].imshow(decoded_imgs_denoise[i], cmap='gray', vmin=0, vmax=1)
    axes[2, i].axis('off')
plt.savefig('plot4_denoise.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

noise_levels = [0.1, 0.2, 0.3]
noise_results = []
for nl in noise_levels:
    test_n = np.clip(x_test_conv + nl * np.random.normal(loc=0.0, scale=1.0, size=x_test_conv.shape), 0., 1.)
    dec = denoise_ae.predict(test_n).reshape(-1, 28, 28)
    m, ma, s = compute_metrics(x_test, dec)
    noise_results.append((nl, m, ma, s))

fig, ax1 = plt.subplots(figsize=(8, 6))
ax2 = ax1.twinx()
ax1.plot(noise_levels, [nr[1] for nr in noise_results], 'b-o', label='MSE')
ax1.plot(noise_levels, [nr[2] for nr in noise_results], 'g-s', label='MAE')
ax2.plot(noise_levels, [nr[3] for nr in noise_results], 'r-^', label='SSIM')
ax1.set_xlabel('Noise Level (std)', fontweight='bold')
ax1.set_ylabel('Error (MSE/MAE)', fontweight='bold')
ax2.set_ylabel('SSIM', fontweight='bold')
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes, prop={'weight': 'bold'})
plt.savefig('plot5_noise_quality.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# --- Model 4: VAE ---
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

latent_dim = 2
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder_vae = models.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
decoder_vae = models.Model(latent_inputs, decoder_outputs, name="decoder")

class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")
    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)))
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"loss": self.total_loss_tracker.result(), "reconstruction_loss": self.reconstruction_loss_tracker.result(), "kl_loss": self.kl_loss_tracker.result()}
    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)))
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        total_loss = reconstruction_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"loss": self.total_loss_tracker.result(), "reconstruction_loss": self.reconstruction_loss_tracker.result(), "kl_loss": self.kl_loss_tracker.result()}

vae = VAE(encoder_vae, decoder_vae)
vae.compile(optimizer=tf.keras.optimizers.Adam())

print("Training VAE...")
start_time = time.time()
history_vae = vae.fit(x_train_conv, epochs=20, batch_size=128, validation_data=(x_test_conv, x_test_conv), verbose=1)
print(f"VAE trained in {time.time() - start_time:.2f}s")


In [ ]:
# Plot 9, 6, 7, 8 for VAE
plt.figure(figsize=(8, 6))
plt.plot(history_vae.history['loss'], label='Training Loss')
plt.plot(history_vae.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.legend(prop={'weight': 'bold'})
plt.savefig('plot9_vae_loss.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

z_m, _, _ = vae.encoder.predict(x_test_conv)
plt.figure(figsize=(8, 6))
plt.scatter(z_m[:, 0], z_m[:, 1], c=y_test, cmap='tab10', s=10)
plt.colorbar()
plt.xlabel('z1', fontweight='bold')
plt.ylabel('z2', fontweight='bold')
plt.savefig('plot6_vae_latent.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

n = 5
random_z = np.random.normal(size=(n * n, latent_dim))
generated_imgs = vae.decoder.predict(random_z)
fig, axes = plt.subplots(n, n, figsize=(6, 6))
for i in range(n):
    for j in range(n):
        axes[i, j].imshow(generated_imgs[i * n + j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[i, j].axis('off')
plt.savefig('plot7_vae_random.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

z_A = z_m[0]
z_B = z_m[1]
alphas = np.linspace(0, 1, 10)
interpolated_z = np.array([(1-a)*z_A + a*z_B for a in alphas])
interpolated_imgs = vae.decoder.predict(interpolated_z)
fig, axes = plt.subplots(1, 10, figsize=(20, 2))
for i in range(10):
    axes[i].imshow(interpolated_imgs[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[i].axis('off')
plt.savefig('plot8_vae_interp.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# Plot 10: Reconstruction Error Distribution & High Error Images
errors = np.mean((x_test - decoded_imgs_fc)**2, axis=(1,2))
plt.figure(figsize=(8, 6))
plt.hist(errors, bins=50)
plt.xlabel('Reconstruction Error (MSE)', fontweight='bold')
plt.ylabel('Frequency', fontweight='bold')
plt.savefig('plot10_error_dist.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()

high_error_idx = np.argsort(errors)[-5:]
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, idx in enumerate(high_error_idx):
    axes[0, i].imshow(x_test[idx], cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Sample {idx}', fontweight='bold')
    axes[1, i].imshow(decoded_imgs_fc[idx], cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
plt.savefig('high_error_imgs.eps', format='eps', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# Additional Latent Dimension Study
latent_dims = [2, 8, 16, 32]
ld_results = []
for ld in latent_dims:
    inp = layers.Input(shape=(784,))
    e = layers.Dense(128, activation='relu')(inp)
    e = layers.Dense(32, activation='relu')(e)
    l = layers.Dense(ld, activation='relu')(e)
    d = layers.Dense(32, activation='relu')(l)
    d = layers.Dense(128, activation='relu')(d)
    out = layers.Dense(784, activation='sigmoid')(d)
    m_ld = models.Model(inp, out)
    m_ld.compile(optimizer='adam', loss='binary_crossentropy')
    m_ld.fit(x_train_flat, x_train_flat, epochs=5, batch_size=128, verbose=0)
    dec = m_ld.predict(x_test_flat).reshape(-1, 28, 28)
    mse, mae, ssim_val = compute_metrics(x_test, dec)
    ld_results.append((ld, mse, ssim_val, m_ld.count_params()))
    print(f"Latent Dim {ld}: MSE={mse:.4f}, SSIM={ssim_val:.4f}, Params={m_ld.count_params()}")
